# 🚀 Week 6 · Day 2 — Model Deployment Prep
**BinX Tech — AI & Machine Learning Internship Program**
**Phase 3 · Sprint 1 · Task 2**

## 🎯 Goal
Load the already-trained, already-tuned Cardiac Patient Monitoring pipeline from Week 4,
verify it works correctly on fresh input, and build the core prediction function that the
Sprint 1 Streamlit app (Task 2 of the backlog) will wrap. This notebook is the **prototype**
step before the actual `app.py` — everything here gets reused directly inside the app.


## 0️⃣ Imports

In [8]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42


## 1️⃣ Load the Saved Pipeline 📦

The pipeline was saved at the end of the Week 4 capstone notebook, and includes a custom
feature-engineering step (`engineer_features`). Because that function isn't a built-in
Scikit-learn class, Python needs the **exact same function defined** in this file before
`joblib.load()` can reconstruct the pipeline — otherwise it can't resolve what that step
actually does.

In [9]:
def engineer_features(X):
    X = X.copy()
    X["HR_Reserve_Ratio"] = X["MaxHR"] / (220 - X["Age"])
    X["High_Chol_Older_Patient"] = ((X["Cholesterol"] > 240) & (X["Age"] > 50)).astype(int)
    return X

model = joblib.load("tuned_cardiac_pipeline.joblib")
print("✅ Pipeline loaded successfully")
print("Pipeline steps:", [name for name, _ in model.steps])

✅ Pipeline loaded successfully
Pipeline steps: ['feature_eng', 'preprocess', 'model']


## 2️⃣ Verify It Still Works — Test on Real Patients 🧪

Before wrapping this in a web app, we sanity-check it on a handful of real rows from the
original dataset, to confirm the loaded pipeline behaves identically to how it did at the
end of training.

In [10]:
df = pd.read_csv("heart.csv")
sample_patients = df.drop(columns=["HeartDisease"]).head(5)
true_labels = df["HeartDisease"].head(5)

predictions = model.predict(sample_patients)
probabilities = model.predict_proba(sample_patients)[:, 1]

results = sample_patients.copy()
results["True Label"] = true_labels.values
results["Predicted"] = predictions
results["Probability (Disease)"] = np.round(probabilities, 3)
results[["Age", "Sex", "ChestPainType", "MaxHR", "True Label", "Predicted", "Probability (Disease)"]]

,Age,Sex,ChestPainType,MaxHR,True Label,Predicted,Probability (Disease)
0,40,M,ATA,172,0,0,0.051
1,49,F,NAP,156,1,0,0.418
2,37,M,ATA,98,0,0,0.175
3,48,F,ASY,108,1,1,0.762
4,54,M,NAP,122,0,0,0.095


**Sanity check passed** if the predictions above look reasonable (they don't need to be
100% correct on 5 random rows — the model's overall test accuracy is 84.8%, so occasional
misses on individual patients are expected and healthy, not a bug).

## 3️⃣ Build a Single-Patient Prediction Function 🩺

This is the exact function the Streamlit app will call every time a user fills out the
form and clicks "Predict." It takes raw feature values (the same 11 columns a clinician
would enter), wraps them into a one-row DataFrame, and returns both the prediction and
the probability.

In [11]:
def predict_patient(age, sex, chest_pain_type, resting_bp, cholesterol, fasting_bs,
                     resting_ecg, max_hr, exercise_angina, oldpeak, st_slope):
    """
    Returns (prediction, probability) for a single patient.
    prediction: 1 = Heart Disease likely, 0 = No Heart Disease
    probability: model's confidence that HeartDisease = 1
    """
    patient = pd.DataFrame([{
        "Age": age,
        "Sex": sex,
        "ChestPainType": chest_pain_type,
        "RestingBP": resting_bp,
        "Cholesterol": cholesterol,
        "FastingBS": fasting_bs,
        "RestingECG": resting_ecg,
        "MaxHR": max_hr,
        "ExerciseAngina": exercise_angina,
        "Oldpeak": oldpeak,
        "ST_Slope": st_slope,
    }])
    pred = model.predict(patient)[0]
    proba = model.predict_proba(patient)[0, 1]
    return pred, proba

In [12]:
# Test it with one realistic patient
pred, proba = predict_patient(
    age=58, sex="M", chest_pain_type="ASY", resting_bp=140, cholesterol=289,
    fasting_bs=0, resting_ecg="Normal", max_hr=110, exercise_angina="Y",
    oldpeak=1.5, st_slope="Flat"
)
print(f"Prediction: {'Heart Disease likely' if pred == 1 else 'No Heart Disease'}")
print(f"Probability of disease: {proba:.1%}")

Prediction: Heart Disease likely
Probability of disease: 94.2%


## 4️⃣ Handle Edge Cases — What the App Needs to Guard Against ⚠️

A web form can't guarantee clean input the way a CSV file can. Let's check what happens
with a couple of edge cases the Streamlit app will need to handle gracefully.

In [21]:
# Edge case 1: Cholesterol = 0 (the exact invalid-data issue from Week 4)
# The pipeline's imputer should absorb this automatically since it was trained on data with the same issue.
edge_pred, edge_proba = predict_patient(
    age=45, sex="F", chest_pain_type="ATA", resting_bp=130, cholesterol=0,
    fasting_bs=0, resting_ecg="Normal", max_hr=150, exercise_angina="N",
    oldpeak=0.0, st_slope="Up"
)
print(f"Cholesterol=0 edge case handled: Prediction={edge_pred}, Probability={edge_proba:.1%}")
print("✅ No crash — the pipeline's median imputer absorbs this the same way it did during training.")

Cholesterol=0 edge case handled: Prediction=0, Probability=4.2%
✅ No crash — the pipeline's median imputer absorbs this the same way it did during training.


**Why this matters:** the Streamlit app (Task 3 in the backlog) will let users type in
any number, including invalid ones like `Cholesterol = 0`. Because our leak-free Pipeline
from Week 4 already includes the `SimpleImputer` as a proper pipeline step — not a one-off
script — it automatically treats a new `0` the same way it learned to treat one during
training. This is a direct payoff of building it as a Pipeline in the first place.

## 5️⃣ Draft the Streamlit App Code 💻

This is the actual code that will become `app.py` in Sprint 1, Task 3. We write it here
first as a string / preview so the logic is verified before it becomes a standalone script
outside the notebook.

In [22]:
streamlit_app_code = '''
import streamlit as st
import pandas as pd
import joblib

def engineer_features(X):
    X = X.copy()
    X["HR_Reserve_Ratio"] = X["MaxHR"] / (220 - X["Age"])
    X["High_Chol_Older_Patient"] = ((X["Cholesterol"] > 240) & (X["Age"] > 50)).astype(int)
    return X

st.set_page_config(page_title="Cardiac Risk Screening (Educational Demo)", page_icon="")
st.title(" Cardiac Patient Monitoring System")
st.caption("Educational ML demo — not a diagnostic tool. No medical advice is provided.")

@st.cache_resource
def load_model():
    return joblib.load("tuned_cardiac_pipeline.joblib")

model = load_model()

col1, col2 = st.columns(2)
with col1:
    age = st.number_input("Age", 18, 100, 50)
    sex = st.selectbox("Sex", ["M", "F"])
    chest_pain_type = st.selectbox("Chest Pain Type", ["ATA", "NAP", "ASY", "TA"])
    resting_bp = st.number_input("Resting Blood Pressure", 80, 220, 130)
    cholesterol = st.number_input("Cholesterol", 0, 600, 200)
    fasting_bs = st.selectbox("Fasting Blood Sugar > 120", [0, 1])
with col2:
    resting_ecg = st.selectbox("Resting ECG", ["Normal", "ST", "LVH"])
    max_hr = st.number_input("Max Heart Rate", 60, 220, 150)
    exercise_angina = st.selectbox("Exercise Angina", ["N", "Y"])
    oldpeak = st.number_input("Oldpeak", -3.0, 7.0, 0.0, step=0.1)
    st_slope = st.selectbox("ST Slope", ["Up", "Flat", "Down"])

if st.button("Predict"):
    patient = pd.DataFrame([{
        "Age": age, "Sex": sex, "ChestPainType": chest_pain_type,
        "RestingBP": resting_bp, "Cholesterol": cholesterol, "FastingBS": fasting_bs,
        "RestingECG": resting_ecg, "MaxHR": max_hr, "ExerciseAngina": exercise_angina,
        "Oldpeak": oldpeak, "ST_Slope": st_slope,
    }])
    pred = model.predict(patient)[0]
    proba = model.predict_proba(patient)[0, 1]

    if pred == 1:
        st.error(f" Model flags elevated risk — probability: {proba:.1%}")
    else:
        st.success(f" Model does not flag elevated risk — probability: {proba:.1%}")
    st.caption("This is an educational demo only. Always consult a qualified clinician.")
'''

with open("app.py", "w") as f:
    f.write(streamlit_app_code)

print(" app.py written — ready for Task 3 (deployment) in the Sprint 1 backlog")

 app.py written — ready for Task 3 (deployment) in the Sprint 1 backlog


In [19]:
import os
print("Files ready in this folder:")
for f in os.listdir("."):
    print(" -", f)

Files ready in this folder:
 - .ipynb_checkpoints
 - app.py
 - day2.ipynb
 - heart.csv
 - tuned_cardiac_pipeline.joblib


## 6️⃣ Reflection ✍️

**Write your own answer here.** Guiding questions:
- 🔒 Why did the `Cholesterol = 0` edge case not crash the app, even though it's the same invalid-data pattern discovered back in Week 4?
- 🩺 What's the difference between what this notebook does and what the actual Streamlit app will do — why prototype the prediction function here first?
- ⚠️ What other edge cases (e.g. missing fields, unexpected category values) should the real app guard against before Sprint 1 is considered done?


## ✅ Deliverable Checklist (Sprint 1, Task 2)
- [ ] 📦 Saved pipeline loaded successfully with the custom feature-engineering function redefined
- [ ] 🧪 Predictions verified against real patient rows from the training data
- [ ] 🩺 Single-patient prediction function built and tested
- [ ] ⚠️ Edge case (Cholesterol = 0) tested and confirmed handled gracefully by the pipeline
- [ ] 💻 Streamlit `app.py` drafted and saved, ready for Task 3 (actual deployment)
- [ ] 🚀 Notebook + `app.py` committed to GitHub with a clear commit message
